<a href="https://colab.research.google.com/github/AmanuelDaget/codealpha_tasks/blob/main/Emotion_Recognition_from_Speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install dependencies**

In [ ]:
# pip install librosa soundfile scikit-learn tensorflow numpy matplotlib seaborn

**Import Libraries**

In [1]:
import os
import numpy as np
import librosa
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, LSTM, Dense, Dropout,
                                      BatchNormalization, Input, Bidirectional)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from google.colab import drive

**Mount drive**

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


**Set your dataset path here**

In [3]:
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/CodeAlpha/Tess_Dataset"
SAVE_PATH    = "/content/drive/MyDrive/Colab Notebooks/CodeAlpha/Emotion Recognition/tess_model"
os.makedirs(SAVE_PATH, exist_ok=True)


**TESS LABEL PARSING**

In [5]:
VALID_EMOTIONS = {'angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad'}
EMOTION_RENAME = {'ps': 'pleasant_surprise'}

all_files  = []
all_labels = []

for root, _, files in os.walk(DATASET_PATH):
    for fname in sorted(files):
        if not fname.lower().endswith('.wav'):
            continue
        parts   = os.path.splitext(fname)[0].lower().split('_')
        emotion = parts[-1]
        if emotion not in VALID_EMOTIONS:
            continue
        emotion = EMOTION_RENAME.get(emotion, emotion)
        all_files.append(os.path.join(root, fname))
        all_labels.append(emotion)

all_files  = np.array(all_files)
all_labels = np.array(all_labels)

print(f"✅ Found {len(all_files)} WAV files\n")
unique, counts = np.unique(all_labels, return_counts=True)
for e, c in zip(unique, counts):
    print(f"   {e:<22} {c} files")

✅ Found 5600 WAV files

   angry                  800 files
   disgust                800 files
   fear                   800 files
   happy                  800 files
   neutral                800 files
   pleasant_surprise      800 files
   sad                    800 files


**FEATURE EXTRACTION**

In [ ]:
MAX_LEN = 200
HOP     = 512

X_raw   = []
y_raw   = []
skipped = 0

for idx, (fpath, emotion) in enumerate(zip(all_files, all_labels)):

    # ── Load audio ──────────────────────────────────────────
    try:
        y_audio, sr = librosa.load(fpath, sr=22050, duration=3.5)
    except Exception as e:
        print(f"  ⚠ Skip {os.path.basename(fpath)}: {e}")
        skipped += 1
        continue

    # ── Preprocessing ────────────────────────────────────────
    y_audio, _ = librosa.effects.trim(y_audio, top_db=20)                        # silence removal
    y_audio    = np.append(y_audio[0], y_audio[1:] - 0.97 * y_audio[:-1])       # pre-emphasis

    # ── Feature extraction ───────────────────────────────────
    mfcc   = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=40, hop_length=HOP)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    chroma = librosa.feature.chroma_stft(y=y_audio, sr=sr, n_chroma=12, hop_length=HOP)
    mel    = librosa.power_to_db(
                 librosa.feature.melspectrogram(y=y_audio, sr=sr, n_mels=40, hop_length=HOP),
                 ref=np.max)
    zcr    = librosa.feature.zero_crossing_rate(y_audio, hop_length=HOP)
    rms    = librosa.feature.rms(y=y_audio, hop_length=HOP)

    # ── Stack → (T, 174) ─────────────────────────────────────
    feat = np.vstack([mfcc, delta, delta2, chroma, mel, zcr, rms]).T.astype(np.float32)

    # ── Pad / truncate to MAX_LEN ────────────────────────────
    T = feat.shape[0]
    if T < MAX_LEN:
        feat = np.vstack([feat, np.zeros((MAX_LEN - T, feat.shape[1]), dtype=np.float32)])
    else:
        feat = feat[:MAX_LEN]

    X_raw.append(feat)
    y_raw.append(emotion)

    if (idx + 1) % 200 == 0:
        print(f"  ... processed {idx+1}/{len(all_files)}")

X_raw = np.array(X_raw, dtype=np.float32)
y_raw = np.array(y_raw)

print(f"\n✅ Feature matrix : {X_raw.shape}   (samples, time, features)")
print(f"   Skipped        : {skipped}")

  ... processed 200/5600
  ... processed 400/5600


**AUDIO AUGMENTATION**

In [ ]:
X_aug_list = list(X_raw)     # start with originals
y_aug_list = list(y_raw)

for idx, (fpath, emotion) in enumerate(zip(all_files, y_raw[:len(all_files)])):

    try:
        y_audio, sr = librosa.load(fpath, sr=22050, duration=3.5)
    except:
        continue

    y_audio, _ = librosa.effects.trim(y_audio, top_db=20)
    y_audio    = np.append(y_audio[0], y_audio[1:] - 0.97 * y_audio[:-1])

    aug_waves = []

    # Variant 1 — Gaussian noise
    aug_waves.append(y_audio + np.random.normal(0, 0.004, len(y_audio)).astype(np.float32))

    # Variant 2 — Time stretch (slow by 10%)
    try:    aug_waves.append(librosa.effects.time_stretch(y_audio, rate=0.9))
    except: pass

    # Variant 3 — Pitch shift (+2 semitones)
    try:    aug_waves.append(librosa.effects.pitch_shift(y_audio, sr=sr, n_steps=2))
    except: pass

    for y_aug in aug_waves:
        # Same feature pipeline as Cell 4
        y_aug  = np.append(y_aug[0], y_aug[1:] - 0.97 * y_aug[:-1])
        mfcc   = librosa.feature.mfcc(y=y_aug, sr=sr, n_mfcc=40, hop_length=HOP)
        delta  = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        chroma = librosa.feature.chroma_stft(y=y_aug, sr=sr, n_chroma=12, hop_length=HOP)
        mel    = librosa.power_to_db(
                     librosa.feature.melspectrogram(y=y_aug, sr=sr, n_mels=40, hop_length=HOP),
                     ref=np.max)
        zcr    = librosa.feature.zero_crossing_rate(y_aug, hop_length=HOP)
        rms    = librosa.feature.rms(y=y_aug, hop_length=HOP)
        feat   = np.vstack([mfcc, delta, delta2, chroma, mel, zcr, rms]).T.astype(np.float32)
        T = feat.shape[0]
        if T < MAX_LEN:
            feat = np.vstack([feat, np.zeros((MAX_LEN - T, feat.shape[1]), dtype=np.float32)])
        else:
            feat = feat[:MAX_LEN]
        X_aug_list.append(feat)
        y_aug_list.append(emotion)

    if (idx + 1) % 200 == 0:
        print(f"  ... augmented {idx+1}/{len(all_files)}")

X_all = np.array(X_aug_list, dtype=np.float32)
y_all = np.array(y_aug_list)

print(f"\n✅ After augmentation: {X_all.shape}  labels: {y_all.shape}")

**Label Encoding & Feature Normalization**

In [ ]:
le    = LabelEncoder()
y_enc = le.fit_transform(y_all)
y_cat = to_categorical(y_enc)
n_classes = y_cat.shape[1]

print(f"🏷️  {n_classes} classes: {list(le.classes_)}")

# ── StandardScaler across feature dimension ─────────────────
n, t, f   = X_all.shape
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_all.reshape(-1, f)).reshape(n, t, f).astype(np.float32)

print(f"✅ Scaled shape: {X_scaled.shape}")

**Train / Validation / Test Split  (70 / 15 / 15)**

In [ ]:
X_tr, X_te, y_tr, y_te, enc_tr, enc_te = train_test_split(
    X_scaled, y_cat, y_enc,
    test_size=0.15, random_state=42, stratify=y_enc)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr,
    test_size=0.15, random_state=42)

print(f"📦  Train : {X_tr.shape[0]}")
print(f"    Val   : {X_val.shape[0]}")
print(f"    Test  : {X_te.shape[0]}")

**Build CNN + BiLSTM Model**

In [13]:
# ============================================================
# CNN  → extracts local spectro-temporal patterns
# BiLSTM → captures long-range context in both directions
# ============================================================
inp = Input(shape=(t, f))

# CNN Block 1
x = Conv1D(128, 5, padding='same', activation='relu')(inp)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)
x = Dropout(0.3)(x)

# CNN Block 2
x = Conv1D(256, 3, padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)
x = Dropout(0.3)(x)

# CNN Block 3
x = Conv1D(128, 3, padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

# BiLSTM
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Dropout(0.4)(x)
x = Bidirectional(LSTM(64))(x)
x = Dropout(0.4)(x)

# Dense head
x   = Dense(128, activation='relu')(x)
x   = Dropout(0.4)(x)
x   = Dense(64,  activation='relu')(x)
out = Dense(n_classes, activation='softmax')(x)

model = Model(inp, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

**TRAIN**

In [14]:
callbacks = [
    EarlyStopping(patience=15, restore_best_weights=True,
                  monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(patience=7, factor=0.5, min_lr=1e-6,
                      monitor='val_loss', verbose=1)
]

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

**Test**

In [15]:
# ============================================================
# CELL 10 — Evaluate on Test Set
# ============================================================
loss, acc = model.evaluate(X_te, y_te, verbose=0)
print(f"\n🎯 Test Accuracy : {acc*100:.2f}%")
print(f"   Test Loss     : {loss:.4f}")

y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
y_true = np.argmax(y_te, axis=1)

print("\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=le.classes_))

**Plots: Accuracy / Loss / Confusion Matrix**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('TESS Emotion Recognition — Results', fontsize=14)

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True)

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[2])
axes[2].set_title('Confusion Matrix')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/results.png", dpi=150)
plt.show()
print("✅ Plot saved")